# Validación de audit logs y logrotate en Kubernetes

Este notebook comprueba el almacenamiento persistente de auditoría del clúster Vault PR, los montajes de los pods, el audit device de Vault y el sidecar de `logrotate`.

Las comprobaciones normales son de solo lectura.

> **Limitación de Oracle:** el clúster PR usa la imagen estándar de Vault Enterprise. Oracle Database Secrets Engine no se valida aquí porque no están instalados el plugin ni Oracle Instant Client.

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if ENV_FILE:
    load_dotenv(ENV_FILE, override=True)

os.environ.setdefault("KUBE_CONTEXT", "PR")
os.environ.setdefault("VAULT_K8S_NAMESPACE", "vaultpr")
os.environ.setdefault("VAULT_HELM_RELEASE_NAME", "vaultpr")
os.environ.setdefault("VAULT_AUDIT_PATH", "/vault/audit/vault.log")

print(f"Contexto:  {os.environ['KUBE_CONTEXT']}")
print(f"Namespace: {os.environ['VAULT_K8S_NAMESPACE']}")
print(f"Release:   {os.environ['VAULT_HELM_RELEASE_NAME']}")
print(f".env:      {ENV_FILE or 'no encontrado (solo será necesario para consultar Vault)'}")

Contexto:  PR
Namespace: vaultpr
Release:   vaultpr
.env:      /Users/jose/Library/CloudStorage/GoogleDrive-jose.maria.merchan@gmail.com/My Drive/Demo/Mapfre_PoC_Vault/.env


## 1. Contexto, StatefulSet y pods

Verifica que el contexto existe, que el StatefulSet está disponible y que cada pod incluye los contenedores `vault` y `auditlog-rotator`.

In [4]:
%%bash
set -euo pipefail

kubectl config get-contexts "${KUBE_CONTEXT}"
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o wide
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' \
  -o custom-columns='POD:.metadata.name,READY:.status.containerStatuses[*].ready,CONTAINERS:.spec.containers[*].name,RESTARTS:.status.containerStatuses[*].restartCount'

not_ready=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o json | \
  jq '[.items[].status.containerStatuses[] | select(.ready != true)] | length')
test "${not_ready}" -eq 0
echo "OK: todos los contenedores de Vault están Ready."

CURRENT   NAME   CLUSTER   AUTHINFO   NAMESPACE
*         PR     PR        PR         default
NAME      READY   AGE   CONTAINERS               IMAGES
vaultpr   3/3     15h   vault,auditlog-rotator   hashicorp/vault-enterprise:2.0.3-ent,josemerchan/vault-logrotate:0.0.1
POD         READY       CONTAINERS               RESTARTS
vaultpr-0   true,true   vault,auditlog-rotator   0,7
vaultpr-1   true,true   vault,auditlog-rotator   0,0
vaultpr-2   true,true   vault,auditlog-rotator   0,0
OK: todos los contenedores de Vault están Ready.


## 2. PVC de auditoría

Comprueba que existe un PVC de auditoría por réplica, que todos están en estado `Bound` y muestra capacidad, StorageClass y volumen asociado.

In [5]:
%%bash
set -euo pipefail

replicas=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o jsonpath='{.spec.replicas}')
audit_pvcs=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pvc -o json | jq --arg release "${VAULT_HELM_RELEASE_NAME}" \
  '[.items[] | select(.metadata.name | startswith("audit-" + $release + "-"))]')

echo "${audit_pvcs}" | jq -r \
  '["PVC","STATUS","CAPACITY","STORAGE_CLASS","VOLUME"],
   (.[] | [.metadata.name,.status.phase,.status.capacity.storage,.spec.storageClassName,.spec.volumeName]) | @tsv' | column -t

count=$(jq 'length' <<<"${audit_pvcs}")
bound=$(jq '[.[] | select(.status.phase == "Bound")] | length' <<<"${audit_pvcs}")
test "${count}" -eq "${replicas}"
test "${bound}" -eq "${replicas}"
echo "OK: ${bound}/${replicas} PVC de auditoría están Bound."

PVC              STATUS  CAPACITY  STORAGE_CLASS  VOLUME
audit-vaultpr-0  Bound   1Gi       standard       pvc-b9028ff2-e68e-47b8-9cf0-16e9e8f83b09
audit-vaultpr-1  Bound   1Gi       standard       pvc-89e54620-04bb-4912-a3af-20bd7d424cd4
audit-vaultpr-2  Bound   1Gi       standard       pvc-514dfe67-e119-4878-b5c0-6bbe1ca3d798
OK: 3/3 PVC de auditoría están Bound.


## 3. Volúmenes, montajes y process namespace

Valida que Vault y el sidecar comparten `/vault/audit`, que el ConfigMap se monta como `/etc/logrotate.conf` y que `shareProcessNamespace` permite al sidecar enviar `SIGHUP` a Vault.

In [6]:
%%bash
set -euo pipefail

sts=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get statefulset "${VAULT_HELM_RELEASE_NAME}" -o json)

jq -r '.spec.template.spec.containers[] as $c |
  $c.volumeMounts[]? |
  select(.mountPath == "/vault/audit" or .mountPath == "/etc/logrotate.conf") |
  [$c.name,.name,.mountPath,(.subPath // "-")] | @tsv' <<<"${sts}" | \
  { printf 'CONTAINER\tVOLUME\tMOUNT\tSUBPATH\n'; cat; } | column -t

jq -e '.spec.template.spec.shareProcessNamespace == true' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "vault") |
  .volumeMounts[] | select(.name == "audit" and .mountPath == "/vault/audit")] | length == 1' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "auditlog-rotator") |
  .volumeMounts[] | select(.name == "audit" and .mountPath == "/vault/audit")] | length == 1' <<<"${sts}" >/dev/null
jq -e '[.spec.template.spec.containers[] | select(.name == "auditlog-rotator") |
  .volumeMounts[] | select(.name == "logrotate-config" and .mountPath == "/etc/logrotate.conf")] | length == 1' <<<"${sts}" >/dev/null
echo "OK: montajes y process namespace configurados correctamente."

CONTAINER         VOLUME            MOUNT                SUBPATH
vault             audit             /vault/audit         -
auditlog-rotator  logrotate-config  /etc/logrotate.conf  logrotate.conf
auditlog-rotator  audit             /vault/audit         -
OK: montajes y process namespace configurados correctamente.


## 4. Configuración y procesos de logrotate

Muestra la política efectiva, la planificación del sidecar, sus procesos y sus últimos logs. También ejecuta `logrotate` en modo debug, que no modifica los ficheros.

In [7]:
%%bash
set -euo pipefail

pod="${VAULT_HELM_RELEASE_NAME}-0"
echo '--- /etc/logrotate.conf ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- cat /etc/logrotate.conf
echo '--- CRONTAB y procesos ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- sh -c 'echo "CRONTAB=${CRONTAB:-unset}"; ps'
echo '--- logrotate debug (sin cambios) ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  exec "${pod}" -c auditlog-rotator -- logrotate -d /etc/logrotate.conf
echo '--- logs del sidecar ---'
kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  logs "${pod}" -c auditlog-rotator --tail=50
echo 'OK: configuración de logrotate legible y válida.'

--- /etc/logrotate.conf ---
/vault/audit/vault.log {
    rotate 2
    size 1M
    missingok
    notifempty
    compress

    postrotate
        pkill -HUP -x vault
    endscript
}
--- CRONTAB y procesos ---
CRONTAB=*/5 * * * *
PID   USER     TIME  COMMAND
    1 crond     0:00 /pause
   45 crond     0:00 /usr/local/bin/vault-logrotate
  300 crond     0:00 /bin/sh -ec cp /vault/config/extraconfig-from-values.hcl /tmp/storageconfig.hcl; [ -n "${HOST_IP}" ] && sed -Ei "s|HOST_IP|${HOST_IP?}|g" /tmp/storageconfig.hcl; [ -n "${POD_IP}" ] && sed -Ei "s|POD_IP|${POD_IP?}|g" /tmp/storageconfig.hcl; [ -n "${HOSTNAME}" ] && sed -Ei "s|HOSTNAME|${HOSTNAME?}|g" /tmp/storageconfig.hcl; [ -n "${API_ADDR}" ] && sed -Ei "s|API_ADDR|${API_ADDR?}|g" /tmp/storageconfig.hcl; [ -n "${TRANSIT_ADDR}" ] && sed -Ei "s|TRANSIT_ADDR|${TRANSIT_ADDR?}|g" /tmp/storageconfig.hcl; [ -n "${RAFT_ADDR}" ] && sed -Ei "s|RAFT_ADDR|${RAFT_ADDR?}|g" /tmp/storageconfig.hcl; if grep -vE '^[[:space:]]*(#|//)' /tmp/storageconfig


reading config file /etc/logrotate.conf
Reading state from file: /var/lib/logrotate.status
state file /var/lib/logrotate.status does not exist
Allocating hash table for state file, size 64 entries

Handling 1 logs

rotating pattern: /vault/audit/vault.log  1048576 bytes (2 rotations)
empty log files are not rotated, old logs are removed
considering log /vault/audit/vault.log
Creating new state
  Now: 2026-07-29 07:52
  Last rotated at 2026-07-29 07:00
  log does not need rotating (log size is below the 'size' threshold)


--- logs del sidecar ---
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting logrotation
Finished logrotation
Starting 

## 5. Persistencia asociada al pod

Relaciona cada pod con su PVC de auditoría y muestra el uso real del filesystem desde el contenedor Vault.

In [10]:
%%bash
set -euo pipefail

for pod in $(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
  get pods -l 'app.kubernetes.io/name=vault,component=server' -o name | cut -d/ -f2); do
  claim=$(kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" get pod "${pod}" \
    -o json | jq -r '.spec.volumes[] | select(.name == "audit") | .persistentVolumeClaim.claimName')
  printf '%s -> %s\n' "${pod}" "${claim}"
  kubectl --context="${KUBE_CONTEXT}" -n "${VAULT_K8S_NAMESPACE}" \
    exec "${pod}" -c vault -- df -h /vault/audit
done

vaultpr-0 -> audit-vaultpr-0
Filesystem                Size      Used Available Use% Mounted on
/dev/vda1                 1.8T     16.6G      1.7T   1% /home/vault
vaultpr-1 -> audit-vaultpr-1
Filesystem                Size      Used Available Use% Mounted on
/dev/vda1                 1.8T     16.6G      1.7T   1% /dev/termination-log
vaultpr-2 -> audit-vaultpr-2
Filesystem                Size      Used Available Use% Mounted on
/dev/vda1                 1.8T     16.6G      1.7T   1% /home/vault


## CLEAN UP

Este notebook es de solo lectura y no crea recursos. La celda confirma ese alcance y elimina únicamente posibles ficheros temporales locales de la validación.

In [12]:
%%bash
set -euo pipefail
rm -f /tmp/vault-audit-validation-*.json
echo 'Cleanup completado: no había recursos remotos que eliminar.'

Cleanup completado: no había recursos remotos que eliminar.
